Comparing Gemini annotations with BiomedParse pseudo-labels
1. Load the existing BiomedParse pseudo-label masks.
2. Identify all connected components without size filtering.
3. Calculate the area, equivalent diameter, maximum and minimum diameters, and bounding-box width and height.
4. Save the measurements to a CSV file.
5. Add Gemini measurements and compare the results across models.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import json

drive.mount("/content/drive")

# Directory containing the foundation model experiment results
PROJECT_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"
PREDICTIONS_DIR = PROJECT_DIR / "predictions"

# Load the dataset manifest
manifest = pd.read_csv(MANIFEST_PATH)

# Select the 10 pilot images
pilot = manifest[
    manifest["is_pilot"].astype(str).str.lower() == "true"
].copy()

pilot = pilot.sort_values("case_id").reset_index(drop=True)

# Verify that all pilot annotations are available
assert len(pilot) == 10

for case_id in pilot["case_id"]:
    prediction_path = PREDICTIONS_DIR / f"{case_id}.json"
    assert prediction_path.exists(), f"Missing prediction: {case_id}"

print("Total dataset images:", len(manifest))
print("Pilot images:", len(pilot))
print("Available Gemini annotations:", len(pilot))

display(
    pilot[
        ["case_id", "category", "split", "is_pilot"]
    ]
)

Mount Google Drive and load the dataset manifest for pilot evaluation

In [ ]:
from PIL import Image
import numpy as np

for _, row in pilot.iterrows():

    case_id = row["case_id"]

    image = np.array(Image.open(row["image_path"]))
    mask = np.array(Image.open(row["pseudo_label_path"]))

    # Check spatial dimensions
    assert image.shape[:2] == mask.shape[:2], (
        f"Dimension mismatch: {case_id}"
    )

    unique_values = np.unique(mask)

    print(
        f"{case_id}: "
        f"image={image.shape}, "
        f"mask={mask.shape}, "
        f"labels={unique_values}"
    )

Inspect the BiomedParse pseudo-label masks and verify their dimensions

In [ ]:
from skimage import measure
import numpy as np
from PIL import Image
import pandas as pd

def measure_pseudo_label_regions(mask, case_id):
    """
    Extract all 8-connected foreground components from a binary mask.
    No minimum area filtering is applied.
    """

    binary_mask = mask == 1

    # 8-connectivity for 2D images
    labeled_mask = measure.label(
        binary_mask,
        connectivity=2
    )

    regions = measure.regionprops(labeled_mask)

    records = []

    for region in regions:

        min_row, min_col, max_row, max_col = region.bbox

        records.append({
            "case_id": case_id,
            "model": "BiomedParse",
            "region_id": region.label,
            "area_px2": region.area,
            "equivalent_diameter_px": region.equivalent_diameter_area,
            "major_axis_length_px": region.axis_major_length,
            "minor_axis_length_px": region.axis_minor_length,
            "bbox_width_px": max_col - min_col,
            "bbox_height_px": max_row - min_row,
            "bbox_xmin": min_col,
            "bbox_ymin": min_row,
            "bbox_xmax": max_col,
            "bbox_ymax": max_row
        })

    return records

Extract all connected components and compute geometric measurements from BiomedParse pseudo-labels

In [ ]:
all_measurements = []

for _, row in pilot.iterrows():

    case_id = row["case_id"]

    mask = np.array(
        Image.open(row["pseudo_label_path"])
    )

    measurements = measure_pseudo_label_regions(
        mask,
        case_id
    )

    all_measurements.extend(measurements)

    print(
        f"{case_id}: {len(measurements)} "
        "connected components"
    )

biomedparse_measurements = pd.DataFrame(all_measurements)

print("\nTotal components:",
      len(biomedparse_measurements))

display(biomedparse_measurements.head(10))

Measure all BiomedParse pseudo-label regions in the pilot dataset

In [ ]:
EVALUATION_DIR = PROJECT_DIR / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

output_path = (
    EVALUATION_DIR /
    "biomedparse_pilot_measurements.csv"
)

biomedparse_measurements.to_csv(
    output_path,
    index=False
)

print("Measurements saved:", output_path)

Save the BiomedParse region measurements for subsequent model comparison

In [ ]:
from PIL import Image
import pandas as pd
import json

gemini_records = []

for _, row in pilot.iterrows():
    case_id = row["case_id"]

    with open(PREDICTIONS_DIR / f"{case_id}.json", "r", encoding="utf-8") as f:
        result = json.load(f)

    annotation = result["annotation"]

    with Image.open(row["image_path"]) as img:
        width, height = img.size

    for region_id, region in enumerate(annotation["regions"], start=1):
        x1, y1, x2, y2 = region["bbox"]

        # Convert normalized coordinates to original image pixels.
        xmin = x1 / 1000 * width
        ymin = y1 / 1000 * height
        xmax = x2 / 1000 * width
        ymax = y2 / 1000 * height

        bbox_width = xmax - xmin
        bbox_height = ymax - ymin

        gemini_records.append({
            "case_id": case_id,
            "model": result["model"],
            "region_id": region_id,
            "bbox_xmin": xmin,
            "bbox_ymin": ymin,
            "bbox_xmax": xmax,
            "bbox_ymax": ymax,
            "bbox_width_px": bbox_width,
            "bbox_height_px": bbox_height,
            "bbox_area_px2": bbox_width * bbox_height,
            "shape": region["shape"],
            "appearance": region["appearance"]
        })

gemini_measurements = pd.DataFrame(gemini_records)

print("Total Gemini regions:", len(gemini_measurements))
display(gemini_measurements.head(10))

Extract pixel-based bounding-box measurements from the saved Gemini annotations

In [ ]:
gemini_output_path = (
    EVALUATION_DIR / "gemini_pilot_measurements.csv"
)

gemini_measurements.to_csv(
    gemini_output_path,
    index=False
)

print("Gemini measurements saved:", gemini_output_path)

Save Gemini bounding-box measurements for comparison with BiomedParse pseudo-labels